# Configure the model & tokenizer  

We saw in the second [notebook](./02-using-an-slm-with-huggingface.ipynb) how easy it is to load a model & tokenizer: 

> ```python 
> from transformers import AutoModelForCausalLM, AutoTokenizer
> 
> model_name = "microsoft/Phi-3.5-mini-instruct"                 # name of the SLM we wish to use 
> model = AutoModelForCausalLM.from_pretrained(model_name)       # loads the pretrained model 
> tokenizer = AutoTokenizer.from_pretrained(model_name)          # loads the tokenizer for the model 
> ```

However for optimal performance, there are some additional configuration options available. 


## Model configuration 

### Accessing the GPU 

When loading the model, set:

> ```python 
> model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")
> ```

Where `device_map="auto"` automatically places the model on the GPU (if available).   

Although, more commonly one will manually/explicitly place it on Nvidia CUDA: 

> ```python 
> import torch
> 
> device = "cuda" if torch.cuda.is_available() else "cpu"  # Check if GPU is available
> 
> model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)  # Move model to GPU
> ```

### Using lower precision  

If you have a GPU, you can also set `torch_dtype=torch.float16` to ensure the model runs in half precision for faster inference:

> ```python 
> model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
> ```

If you're using a CPU, using `torch.bfloat16` actually provides better precision & performance. 


### Putting it together 

Loading our model configuration from YAML: 

In [ ]:

import yaml
import torch

from transformers import AutoModelForCausalLM


# Specify the model name to load. This is the pre-trained Hugging Face model to be used for causal language modeling.
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"


def get_device():
    """Determine the available device for computation."""
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():     # macOS arm 
        return "mps"
    return "cpu"


def load_model_config():
    """
    Loading a model for use
    """
    with open("./finetuning_params.yml", "r") as file:
        config = yaml.safe_load(file)
    return config["model_kwargs"]


model_kwargs = load_model_config()
print(model_kwargs)


In [ ]:
device = get_device()

model_kwargs = load_model_config()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs).to(device)


### Understanding the configuration options 

When loading a Hugging Face model, setting these parameters controls various aspects of model behavior and performance:

`use_cache: False`
- Controls whether key-value cache is used during inference.
- Setting it to False ensures that key-value caching is disabled, which is useful in cases like training or sequence-to-sequence tasks (e.g., translation) where caching might not be beneficial.
- If True, it can speed up decoding for autoregressive models by reusing past attention states.

`trust_remote_code: True`
- Allows execution of custom model code from the Hugging Face repository.
- Some models include special optimizations or modifications beyond the standard Transformer implementations, and this enables those custom implementations.
- Security risk: If the repository isn’t trusted, enabling this could execute arbitrary code.

`torch_dtype: bfloat16`
- Specifies the data type for model weights and computations.
- bfloat16 (Brain Floating Point 16) is an efficient format for mixed-precision training and inference, especially on modern GPUs like NVIDIA A100 and H100. Compared to float16, bfloat16 has a wider exponent range, reducing numerical instability issues.
- NOTE: for Apple Silicon, it's better to use torch_dtype: float32 though it requires higher memory usage and slower computation. 

`attn_implementation: flash_attention_2`
- Enables FlashAttention v2, a highly optimized attention mechanism designed to improve memory efficiency and speed.
- This is particularly useful for large models running on GPUs, as it significantly reduces memory usage while maintaining accuracy. Requires a compatible GPU and PyTorch 2.0+. 



### Expected speed improvement  

| Setup                               | Expected Inference Time  |
|-------------------------------------|--------------------------|
| CPU                                 | 30+ seconds              |
| GPU, FP32                           | ~10 seconds              |
| GPU, FP16                           | ~2-3 seconds             |
| GPU, Flash Attention and cache      | ~1 second or less 🚀 |



### 🤔 How does your model's peformance compare? 

Play with the configuration options, and examine the impact on speed.  How does it compare to colleagues? 

How much memory is your system using? 




----

## Configure the tokenizer  

There are also some options you can use to configure the tokenizer, as described in the comments below. 

In [ ]:
from transformers import AutoTokenizer

def create_tokenizer():
    """
    """
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Function calling does not require that many tokens (128k)
    # Set the maximum token length for the tokenizer to prevent unnecessary memory usage.
    # 2048 tokens are sufficient for function calling scenarios.
    tokenizer.model_max_length = 2048

    # Configure the padding token to avoid infinite generation loops:
    # - Use the unknown token (`unk_token`) if available.
    # - Fallback to the end-of-sequence token (`eos_token`) if `unk_token` is unavailable.
    # use unk_token rather than eos_token to prevent endless generation in Phi3
    tokenizer.pad_token = (
        tokenizer.unk_token if tokenizer.unk_token is not None else tokenizer.eos_token
    )

    # Ensure the pad token ID is set correctly for efficient padding operations.
    tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

    # Set the padding side to "right" for efficient tokenization and batch processing.
    # Right-side padding is often preferred for causal language models.
    tokenizer.padding_side = "right"
    return tokenizer


tokenizer = create_tokenizer()